In [1]:
import duckdb

In [2]:
conn = duckdb.connect()
%load_ext sql
%sql conn --alias duckdb

loading the two files created in 01_data_preparation.ipynb into their respective tables

In [3]:
%%sql
CREATE OR REPLACE TABLE team_data AS
SELECT * 
FROM read_parquet('./data/team_data.parquet')

Running query in 'duckdb'

Count
9388


In [4]:
%%sql
CREATE OR REPLACE TABLE  player_data AS
SELECT * 
FROM read_parquet('./data/player_data.parquet')

Running query in 'duckdb'

Count
46940


## Team and Region Performance

The following creates a view to identify the primary league of each team which can be safely assumed by choosing the league they have played the most games in, as there is no possibility for a team to have more games at an international event than in their regional league.

In [5]:
%%sql
CREATE OR REPLACE VIEW v_primary_league AS (
    WITH games AS (
    SELECT 
        teamname,
        league, 
        count(*) as games_in_league
    FROM 
        team_data
    GROUP BY ALL
    )
    SELECT 
        teamname, 
        arg_max(league, games_in_league) as league
    FROM 
        games
    GROUP BY ALL
)

Running query in 'duckdb'

Count


In [6]:
%%sql
SELECT * FROM v_primary_league LIMIT 3

Running query in 'duckdb'

teamname,league
Natus Vincere,LEC
JD Gaming,LPL
Team WE,LPL


This creates a view that shows only games at the 3 international events First Stand, MSI and Worlds. It also includes the region of each team as well as the region of their opponent by joining the primary_league view to enable aggregations based on regions.

In [7]:
%%sql 
CREATE OR REPLACE VIEW v_international_games AS (
    SELECT
        ld.gameid, 
        ld.date,
        ld.league, 
        ld.teamname, 
        pl.league as team_region, 
        opponent, 
        pl_o.league as enemy_region,
        game,
        result,
    FROM 
        team_data ld 
        JOIN v_primary_league pl USING(teamname) 
        JOIN v_primary_league pl_o ON ld.opponent = pl_o.teamname 
    WHERE ld.league IN ('FST', 'MSI', 'WLDs')
)

Running query in 'duckdb'

Count


In [8]:
%%sql
SELECT * FROM v_international_games LIMIT 3

Running query in 'duckdb'

gameid,date,league,teamname,team_region,opponent,enemy_region,game,result
LOLTMNT01_349243,2026-03-16 13:10:29,FST,BNK FEARX,LCK,Bilibili Gaming,LPL,1,0
LOLTMNT01_350155,2026-03-16 14:00:50,FST,Bilibili Gaming,LPL,BNK FEARX,LCK,2,0
LOLTMNT01_349264,2026-03-16 14:46:52,FST,BNK FEARX,LCK,Bilibili Gaming,LPL,3,1


Making use of the new view to create a pivot table showing the winrate by region against every other region

In [9]:
%%sql
PIVOT 
    v_international_games
    ON enemy_region
    USING round(avg(result),3)
GROUP BY 
    team_region
ORDER BY 
    team_region

Running query in 'duckdb'

team_region,CBLOL,LCK,LCP,LCS,LEC,LPL
CBLOL,None,0.0,0.444,0.222,0.4,0.0
LCK,1.0,0.5,0.864,0.857,0.6,0.655
LCP,0.556,0.136,None,0.364,0.423,0.438
LCS,0.778,0.143,0.636,None,0.8,0.25
LEC,0.6,0.4,0.577,0.2,0.5,0.321
LPL,1.0,0.345,0.563,0.75,0.679,0.5


We can see that Europe (LEC) performed the worst against North America (LCS) despite performing better than them against the Asian Top Teams (LPL, LCK). We can also see the dominance of the LCK, being the only region to have a positive winrate against every other region. Another insight from this is that since the start of 2025, there have been no games at international tournaments where LCS, CBLOL or LCP teams have played against their own region. 

Another use of the international games view is to identify how individual teams from a region did against other regions. This query also includes the number of games to allow for fairer comparisons. By using ROLLUP the results also show how all teams did in general as well as how the entire region did.

In [10]:
%%sql
SELECT 
    teamname, 
    enemy_region, 
    avg(result) as winrate, 
    count(*) as games
FROM 
    v_international_games
WHERE 
    team_region = 'LEC'
GROUP BY 
    ROLLUP (teamname, enemy_region)
ORDER BY 
    games DESC
LIMIT 7

Running query in 'duckdb'

teamname,enemy_region,winrate,games
None,None,0.4107142857142857,112
G2 Esports,None,0.45901639344262296,61
Karmine Corp,None,0.39285714285714285,28
G2 Esports,LPL,0.2857142857142857,21
Movistar KOI,None,0.3333333333333333,18
G2 Esports,LCK,0.5882352941176471,17
Karmine Corp,LCK,0.2,10


We see that G2 accounts for more than half of the games of European teams while still having the highest winrate. This confirms a common saying, that Europe is a one team region. G2 Esports' high winrate against LCK teams also stood out, which prompted me to check for non-LCK teams based on their winrate against LCK teams

In [11]:
%%sql
WITH vs_lck AS (
    SELECT 
        teamname, 
        avg(result) as winrate_vs_LCK, 
        count(*) as games
    FROM 
        v_international_games
    WHERE 
        enemy_region = 'LCK' 
        AND team_region != 'LCK'
    GROUP BY 
        teamname
)
SELECT 
    *, 
    RANK() OVER (ORDER BY winrate_vs_LCK DESC) as rank 
FROM 
    vs_lck
QUALIFY 
    rank <= 5


Running query in 'duckdb'

teamname,winrate_vs_LCK,games,rank
G2 Esports,0.5882352941176471,17,1
Bilibili Gaming,0.5,22,2
Anyone's Legend,0.47058823529411764,17,3
Invictus Gaming,0.25,4,4
LYON,0.25,8,4


This confirms that there are only 2 teams outside of the LCK that have a winrate of 50% or higher against them since the beginning of 2025 and that G2 has performed the best amongst them. Furthermore it shows just how big the difference between this performance and the rest of the teams is.

Next I wanted to check the winrate as well as the number of games of all teams at international events. I then ranked them based on their number of games and winrate and filtered to only include teams that are in the top 5 of either category.

In [12]:
%%sql
WITH team_winrate AS (
    SELECT 
        teamname, 
        round(avg(result), 3) as winrate, 
        count(*) as number_of_games
    FROM 
        v_international_games
    GROUP BY 
        teamname
) 
SELECT 
    teamname, 
    winrate, 
    rank() OVER (ORDER BY winrate DESC) as rank_winrate, 
    number_of_games, 
    rank() OVER (ORDER BY number_of_games DESC) as rank_games
FROM 
    team_winrate
QUALIFY
    rank_games <= 5 OR rank_winrate <= 5
ORDER BY 
    rank_games

Running query in 'duckdb'

teamname,winrate,rank_winrate,number_of_games,rank_games
T1,0.662,3,68,1
Bilibili Gaming,0.639,4,61,2
G2 Esports,0.459,10,61,2
Hanwha Life Esports,0.696,2,46,4
Gen.G,0.634,5,41,5
KT Rolster,0.75,1,16,15


The result shows that G2 is the only team within the top 5 by number of games to have a negative winrate, ranking only at 10th place in that category. We also see that 3 Teams (T1, Hanwha, Gen.G) from the LCK are in the top 5 for both categories and fellow LCK team KT Rolster, who only attended one international event in the last year, comes in at first place when ranked by winrate but are down to 15th spot in the number of games ranking. 

## Player Analysis

Next my goals were to see, on a basic level, how players performed so far in 2026:
1. at a specific event (MSI)
2. how many unique champions each player has played

I created a view that aggregates totals and averages per game for each stat as well as a total kill-death-assist ratio (kda). 

In [13]:
%%sql
CREATE OR REPLACE VIEW player_stats AS (
    SELECT 
    playername,
    teamname,
    league,
    date_part('year', date) as year,
    split,
    sum(kills) as t_kills,
    sum(assists) as t_assists,
    sum(deaths) as t_deaths,
    count(playername) as num_games,
    round(avg(kills), 2) as kills_per_game,
    round(avg(assists), 2) as assists_per_game,
    round(avg(deaths), 2) as deaths_per_game,
    round((t_kills + t_assists) / t_deaths, 2) as kda,
FROM 
    player_data 
    JOIN team_data USING(gameid, teamname) 
WHERE 
    date_part('year', date) = '2026' 
GROUP BY 
    ALL
)


Running query in 'duckdb'

Count


In [14]:
%%sql
SELECT * FROM player_stats LIMIT 3

Running query in 'duckdb'

playername,teamname,league,year,split,t_kills,t_assists,t_deaths,num_games,kills_per_game,assists_per_game,deaths_per_game,kda
Fleshy,Team Vitality,LEC,2026,Spring,24,281,81,30,0.8,9.37,2.7,3.77
kyeahoo,Karmine Corp,LEC,2026,Spring,132,286,104,42,3.14,6.81,2.48,4.02
YoungJae,LOUD,CBLOL,2026,Spring,92,170,100,29,3.17,5.86,3.45,2.62


Through this view we can then check who the best performing players at MSI this year were. It should be noted that KDA often times does not tell the whole picture but it does give a general idea of player performance.  

In [15]:
%%sql
SELECT * 
FROM 
    player_stats
WHERE
    league = 'MSI'
    AND year = 2026
ORDER BY
    kda DESC
LIMIT 3

Running query in 'duckdb'

playername,teamname,league,year,split,t_kills,t_assists,t_deaths,num_games,kills_per_game,assists_per_game,deaths_per_game,kda
Knight,Bilibili Gaming,MSI,2026,Spring,93,122,30,17,5.47,7.18,1.76,7.17
Zeka,Hanwha Life Esports,MSI,2026,Spring,105,138,40,20,5.25,6.9,2.0,6.08
Berserker,LYON,MSI,2026,Spring,86,99,34,17,5.06,5.82,2.0,5.44


Next I wanted to see which player played the most unique champions this year. For comparison's sake I have also included the total number of games as well as the positional rank.

In [16]:
%%sql
SELECT 
    playername,
    position,
    count(DISTINCT champion) as unique_champions, 
    count(gameid) number_of_games,
    rank() OVER (PARTITION BY position ORDER BY unique_champions DESC) as rank_in_position
FROM 
    player_data 
    JOIN team_data USING(gameid, teamname)
WHERE
    date >= '2026-01-01'
GROUP BY 
    playername,
    position 
ORDER BY unique_champions DESC, rank_in_position 
LIMIT 10

Running query in 'duckdb'

playername,position,unique_champions,number_of_games,rank_in_position
Dhokla,top,28,86,1
BrokenBlade,top,27,107,2
Knight,mid,25,152,1
Viper,bot,25,152,1
Clear,top,25,97,3
Zeus,top,25,98,3
Keria,sup,24,107,1
Doggo,bot,24,66,2
Siwoo,top,24,99,5
Monki,jng,23,102,1


These results show that this year, toplaners have had the most unique champion picks. It also shows that the number of unique picks does not correlate heavily with the number of games and that overall, positions are relatively equal in the amount of viable champions. I then decided to compare these results to those of 2025 up until the same date.  

In [17]:
%%sql
SELECT 
    playername,
    position,
    count(DISTINCT champion) as unique_champions, 
    count(gameid) number_of_games,
    rank() OVER (PARTITION BY position ORDER BY unique_champions DESC) as rank_in_position
FROM 
    player_data 
    JOIN team_data USING(gameid, teamname)
WHERE
    date BETWEEN '2025-01-01' AND '2025-08-20'
GROUP BY 
    playername,
    position 
ORDER BY unique_champions DESC, rank_in_position 
LIMIT 10

Running query in 'duckdb'

playername,position,unique_champions,number_of_games,rank_in_position
Flandre,top,27,138,1
Knight,mid,26,143,1
ShowMaker,mid,26,96,1
BrokenBlade,top,26,80,2
Myrwn,top,25,75,3
Kiin,top,25,113,3
Chovy,mid,24,113,3
Targamas,sup,23,96,1
Kanavi,jng,23,135,1
Keria,sup,23,110,1


By comparing the two results we can see that midlane had more unique picks in 2025 while toplane remains in a similar spot. Most notably, bot has disappeared from the top 10 in the 2025 results. This is surprising, as traditionally it has been the role with the lowest amount of viable champions but with the current state of the game, 2026 has seen many champions that have not been played in botlane before.

The results of the player query above then led me to check unique picks on a position basis to test my theory that botlane has seen a lot more unique champions this year.

In [18]:
%%sql
WITH unique_champions AS(
SELECT 
    DISTINCT position, 
    date_part('year', date) AS year, 
    count(DISTINCT champion) OVER(PARTITION BY position, year) as unique_champions  
FROM 
    player_data 
    JOIN team_data USING(gameid, teamname) 
ORDER BY 
    year DESC
)
PIVOT 
    unique_champions
ON 
    year
USING 
    SUM(unique_champions)
GROUP BY 
    position


Running query in 'duckdb'

position,2025,2026
mid,62,56
top,69,67
jng,48,49
sup,45,47
bot,34,46


Although botlane is still the position with the least amount of champions picked in 2026, we already have a higher number of picks this year than last year with almost 3 months of play still ahead of us. However I do not expect this number to rise much further, as it is more likely that the game will be patched in a way that favors traditional botlane champions to preserve the identity of the role.

The next query shows the champions that were picked in 2026 but not in 2025, 

In [19]:
%%sql
WITH champs_by_year AS (
    SELECT 
        date_part('year', date) as year, 
        champion 
    FROM 
        player_data 
        JOIN team_data USING(gameid, teamname) 
    WHERE position = 'bot'
),
champ_pivot AS (
    PIVOT 
        champs_by_year
    ON 
        year
    USING 
        count(champion)
    GROUP BY 
        champion
)
SELECT * 
FROM 
    champ_pivot
WHERE 
    "2025" <= 10
ORDER BY 
    "2026" DESC


Running query in 'duckdb'

champion,2025,2026
Mel,5,44
Viktor,1,36
Cassiopeia,0,22
Syndra,0,20
Kog'Maw,8,14
Xerath,0,13
Taliyah,1,8
Seraphine,1,7
Vladimir,0,4
Akali,0,4


With the exception of Kog'Maw, all of the frequently picked champions in this list are Mages and didn't see any or very little play in all of 2025, confirming the theory that a new set of champions is viable currently. 

## Pick and Banrates
For the next Analysis I was interested in pick and banrates of champions in this year. I created views for ban statistics, team composition and pick statistics.

In [29]:
%%sql
CREATE OR REPLACE VIEW v_ban_stats AS (
    WITH banned_champions AS (
        SELECT 
            DISTINCT
            UNNEST(bans) as champion
        FROM 
            team_data
    )
SELECT 
    champion, 
    countif(champion IN bans) as number_of_bans, 
    league, 
    date_part('year', date) as year
FROM 
    team_data 
    JOIN banned_champions ON champion IN bans 
GROUP BY 
    champion, 
    league, 
    year 
)


Running query in 'duckdb'

Count


In [30]:
%sql SELECT * FROM v_ban_stats LIMIT 3

Running query in 'duckdb'

champion,number_of_bans,league,year
Jax,11,LCP,2026
Galio,14,LCS,2026
Zac,1,LCS,2026


In [22]:
%%sql
CREATE OR REPLACE VIEW v_team_comp AS (
    SELECT 
        gameid, 
        teamname,
        league,
        date, 
        list(champion) as comp 
    FROM 
        team_data 
        JOIN player_data USING(gameid, teamname) 
    GROUP BY gameid, teamname, league, date
)

Running query in 'duckdb'

Count


In [23]:
%sql SELECT * FROM v_team_comp LIMIT 3

Running query in 'duckdb'

gameid,teamname,league,date,comp
LOLTMNT05_184106,Movistar KOI,LEC,2026-03-28 20:48:34,"['Gwen', 'Zaahen', 'Sylas', 'Miss Fortune', 'Nautilus']"
LOLTMNT05_183186,Team Heretics,LEC,2026-03-30 19:23:52,"['Sion', 'Xin Zhao', 'Hwei', 'Jinx', 'Lulu']"
LOLTMNT02_375568,HANJIN BRION,LCK,2026-04-01 08:11:43,"['Yorick', 'Jarvan IV', 'Cassiopeia', 'Jhin', 'Neeko']"


In [ ]:
%%sql
CREATE OR REPLACE VIEW v_pick_stats AS (
    WITH picked_champions AS (
        SELECT 
            DISTINCT
            UNNEST(comp) as champion
        FROM 
            v_team_comp 
    )
    SELECT 
        champion, 
        countif(champion IN comp) as picks,
        league,
        date_part('year',date) as year
    FROM 
        v_team_comp 
        JOIN picked_champions ON champion IN comp 
    GROUP BY ALL
)



Running query in 'duckdb'

Count


In [38]:
%%sql
SELECT * FROM v_pick_stats WHERE league = 'LCK' AND year = '2026' ORDER BY picks DESC LIMIT 3

Running query in 'duckdb'

champion,picks,league,year
Jarvan IV,133,LCK,2026
Ryze,125,LCK,2026
Xin Zhao,125,LCK,2026


In [25]:
%sql SELECT * FROM v_pick_stats LIMIT 3

Running query in 'duckdb'

champion,picks,league,year
Lulu,10,MSI,2026
Xin Zhao,58,LCP,2026
Xin Zhao,81,LEC,2025


Using the created views I calculated and combined the total picks as well as the pick / ban percentages for each champion. 

In [ ]:
%%sql
WITH "2026_picks" AS (
    SELECT 
        champion, 
        sum(picks) as total_picks,
        round(sum(picks)  / (SELECT count(DISTINCT gameid) FROM team_data WHERE date >= '2026-01-01'), 2) as pickrate
    FROM v_pick_stats
    WHERE year = 2026 
    GROUP BY champion
    ORDER BY total_picks DESC
),
"2026_bans" AS (
    SELECT 
        champion, 
        sum(number_of_bans) as total_bans
        round(sum(number_of_bans) / (SELECT count(DISTINCT gameid) FROM team_data WHERE date >= '2026-01-01'), 2) as banrate 
    FROM v_ban_stats 
    WHERE year = 2026 
    GROUP BY champion 
    ORDER BY total_bans DESC
)
SELECT 
    *, 
    round((total_picks + total_bans) / (SELECT count(DISTINCT gameid) FROM team_data WHERE date >= '2026-01-01'), 2) as "pick/ban"
FROM 
    '2026_picks' JOIN '2026_bans' USING(champion)
ORDER BY "pick/ban" DESC

Running query in 'duckdb'

champion,total_picks,pickrate,total_bans,banrate,pick/ban
Orianna,349,0.17,1242,0.59,0.76
Varus,334,0.16,1090,0.52,0.68
Rumble,530,0.25,773,0.37,0.62
Jarvan IV,553,0.26,657,0.31,0.58
Nocturne,286,0.14,853,0.41,0.54
Jayce,343,0.16,754,0.36,0.52
Vi,453,0.22,596,0.28,0.5
Nautilus,374,0.18,663,0.32,0.49
Bard,428,0.2,587,0.28,0.48
Ryze,575,0.27,434,0.21,0.48


Through this query we can see that there are 3 champions that have been either picked or banned in more than 60% of games, which is a very high percentage but not unusual for strong picks. Keep in mind that a champion can only be picked once per Series which is why pickrates are rather low all around. 

## Bonus Query

Query to answer a question found on Reddit after a match. By using the string_agg function I have grouped all champions by team for each game. I have then calculated the number of wins using the countif function on the result value for wins and losses and calculated the winrate. I then used the where clause to pick only games where 1. the specified champions are present in the same team composition 2. the games took place in the major leagues 3. the games are part of the current summer split.

In [40]:
%%sql
WITH team_comp AS (
    SELECT 
        gameid, 
        teamname, 
        string_agg(champion, ', ') as comp 
    FROM 
        team_data 
        JOIN player_data USING(gameid, teamname) 
    GROUP BY gameid, teamname
)
SELECT 
    countif(result = 1) as wins,
    countif(result = 0) as losses,
    avg(result) as winrate 
FROM 
    v_team_comp 
    JOIN team_data USING(gameid, teamname) 
WHERE 
    'Lucian' IN comp AND 'Milio' IN comp 
    AND v_team_comp.league IN ('MSI', 'WLDs', 'FST', 'LCK', 'LPL', 'LEC', 'LCS', 'LCP') 
    AND patch > '16.13'


Running query in 'duckdb'

wins,losses,winrate
83,123,0.4029126213592233
